In [53]:
# basic imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# model imports
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# metric imports
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error

# model selection imports
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_friedman1, make_regression
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    fbeta_score,
)

# preprocessing imports
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [44]:
fraud_train = pd.read_parquet(
    "https://lab.cs307.org/fraud/data/fraud-train.parquet",
)
fraud_test = pd.read_parquet(
    "https://lab.cs307.org/fraud/data/fraud-test.parquet",
)

# create X and y for train
X_train = fraud_train.drop("Fraud", axis=1)
y_train = fraud_train["Fraud"]

# create X and y for test
X_test = fraud_test.drop("Fraud", axis=1)
y_test = fraud_test["Fraud"]

In [45]:
fraud_train.shape
fraud_train.head()

,PC01,PC02,PC03,PC04,PC05,PC06,PC07,PC08,PC09,PC10,...,PC21,PC22,PC23,PC24,PC25,PC26,PC27,PC28,Amount,Fraud
0,-0.514509,0.899378,1.627215,-0.142250,0.005250,-0.235422,0.482540,0.247403,-0.562327,-0.166813,...,-0.143290,-0.390205,0.030719,0.184779,-0.348711,0.073253,0.273217,0.107938,3.59,0
1,-0.813568,-0.373893,1.152977,-0.449774,-3.868866,2.780636,3.654192,-0.672442,0.753230,-0.662803,...,-0.376783,-0.004239,0.074801,0.124238,-0.448493,0.861423,-0.093639,-0.711632,798.01,0
2,-2.443142,3.258831,-0.791511,0.223548,0.007932,-1.263044,1.220214,-0.418068,1.860453,4.184883,...,-0.348587,0.531679,0.058990,0.371638,-0.207398,-0.505837,0.524542,-0.343895,1.79,0
3,-0.397300,0.922104,1.224699,-0.334974,0.322603,-0.117372,0.534683,0.175550,-0.486404,-0.120147,...,-0.239303,-0.695001,-0.128231,-0.536463,-0.138971,0.107526,0.255644,0.100814,2.69,0
4,1.994046,-0.367813,-0.462867,0.338661,-0.485326,-0.241576,-0.590987,0.089319,1.413224,-0.149292,...,-0.196388,-0.484457,0.421867,0.601393,-0.448014,-0.646256,0.027632,-0.027244,4.49,0


In [46]:
(fraud_train['Fraud']).value_counts()

Fraud
0    53961
1      315
Name: count, dtype: int64

In [47]:
315 / len(fraud_train)

0.005803670130444395

In [48]:
fraud_train['Amount'].agg(['mean','std','max','median'])

mean         88.197903
std         241.535617
max       10199.440000
median       21.690000
Name: Amount, dtype: float64

In [57]:
#train a model 
num_features = ['Amount', 'PC01', 'PC02', 'PC03', 'PC04', 'PC05', 'PC06', 'PC07', 'PC08', 'PC09', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19', 'PC20', 'PC21', 'PC22', 'PC23', 'PC24', 'PC25', 'PC26', 'PC27', 'PC28']

#preprocessing for num features 
numeric_transformer = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

#create general preprocessor 
preprocessor = ColumnTransformer(
    [
        ('numeric', numeric_transformer, num_features)
    ],
    remainder= 'drop',
)

#put everything tgt, build knn model 
tree = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier()),
    ]
)
weights_list = [
    {0: 1, 1: 1},
    {0: 1, 1: 2},
    "balanced",
]

rf_param_grid = {
    "classifier__random_state": [5,20],
    "classifier__max_depth": [2, 20],
    "classifier__class_weight" : weights_list,
}

scoring = {
    "recall": make_scorer(recall_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(fbeta_score, beta=2),
}

rf_grid = GridSearchCV(
    tree,
    rf_param_grid,
    cv=5,
    scoring=scoring,
    refit='f1'
)

rf_grid.fit(X_train, y_train)

y_test_pred = rf_grid.predict(X_test)

# calculate and print the test accuracy
test_prec = precision_score(y_test, y_test_pred)
print(f"Test precision score: {test_prec}")

test_rec = recall_score(y_test, y_test_pred)
print(f"Test recall score: {test_rec}")


Test precision score: 0.9285714285714286
Test recall score: 0.8227848101265823


In [58]:
from joblib import dump
dump(rf_grid, "fraud.joblib")

['fraud.joblib']